# P1 통계 검정과 A/B 표본 수 계산

- 입력: BigQuery 집계 결과(성공 수 / 전체 수). 원본 데이터는 필요 없음
- 판정 기준(사전에 정함): **p < 0.05** 그리고 **상대 차이 10% 이상**
- 두 비율 차이 z-검정은 2×2 카이제곱 검정과 같은 결과(연속성 보정 없음)이며, 방향과 신뢰구간을 함께 보려고 z-검정을 사용

In [ ]:
# 셀 0: 두 비율 차이 검정 함수
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

def compare_rates(name, success_a, n_a, success_b, n_b):
    """그룹 A와 B의 비율을 비교한다. A가 비교 대상, B가 기준."""
    rate_a = success_a / n_a
    rate_b = success_b / n_b
    z, p = proportions_ztest(count=[success_a, success_b], nobs=[n_a, n_b])
    low, high = confint_proportions_2indep(success_a, n_a, success_b, n_b, compare='diff')
    rel_diff = (rate_a / rate_b - 1) * 100
    print(f"{name}")
    print(f"  A {rate_a:.1%} vs B {rate_b:.1%} | 차이 {(rate_a-rate_b)*100:+.1f}%p "
          f"[{low*100:+.1f}, {high*100:+.1f}] | 상대 {rel_diff:+.1f}% | p = {p:.4f}")

## 1. 단계별 시즌(A) vs 비시즌(B) — 최대 도달 기준 (30번 쿼리)

결과: 결제 시작 이전 단계는 유의하게 하락, **결제 정보 → 구매는 차이 없음(p = 0.147)**.
결제 시작 → 결제 정보는 시즌이 더 낮음(시즌 신규 때문, 셀 5).
조회 → 결제 시작(장바구니 무관)도 함께 검정: 18.1% vs 9.5%.

In [ ]:
print("=== 단계별 시즌 vs 비시즌 ===")
compare_rates("방문 → 조회",           28454, 124133, 21514, 107189)
compare_rates("조회 → 구매 의도",       9885,  28454,  4819,  21514)
compare_rates("구매 의도 → 결제 시작",  5154,   9885,  2038,   4819)
compare_rates("결제 시작 → 결제 정보",  3224,   5154,  1482,   2038)
compare_rates("결제 정보 → 구매",       2369,   3224,  1059,   1482)
compare_rates("조회 → 결제 시작 (장바구니 무관)", 5154, 28454, 2038, 21514)


## 2. H3: 비시즌 구매 의도 하락은 신규에서 더 큰가 (31번 쿼리)

두 집단의 **변화량**을 비교하는 질문이라 로지스틱 회귀의 기간 × 신규 상호작용 항으로 검정.
결과: 상호작용 계수 −0.460 (오즈비 0.631), p < 0.001 → **H3 지지**. 칸 4개에 계수 4개인 포화 모델이라 잔차 자유도 0은 정상.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.DataFrame({
    'period':  ['season', 'season', 'off', 'off'],
    'segment': ['new', 'returning', 'new', 'returning'],
    'success': [5689, 4196, 2604, 2215],     # 구매 의도 도달
    'n':       [18500, 9954, 15183, 6331],   # 조회 도달
})
df['rate'] = df['success'] / df['n']

model = smf.glm(
    "rate ~ C(period, Treatment('season')) * C(segment, Treatment('returning'))",
    data=df,
    family=sm.families.Binomial(),
    var_weights=df['n'],
).fit()

print(model.summary())
print(np.exp(model.params))   # 오즈비

## 3. H4: 시즌 신규 결제 정보 전환율, 기기·국가 (34번 쿼리)

결과: 모바일 +3.0% (p = 0.50), 미국 외 +2.3% (p = 0.60), 신뢰구간이 0을 포함하고 방향도 반대 → **H4 기각**

In [ ]:
compare_rates("H4 기기: 모바일 vs 데스크톱 (시즌 신규 결제 정보)", 496, 1085, 714, 1608)
compare_rates("H4 지역: 미국 외 vs 미국", 676, 1493, 556, 1256)

## 4. H4-b: 주별 비교 (35번 쿼리)

블프 주(A) vs 12/14 주(B). 신규 −36.9% (p < 0.0001), 재방문 −7.0% (p = 0.013, 크기 기준 10% 미달).
비율 차이는 유의하지만, 12/14 주의 상승이 실제 개선인지는 셀 4-1에서 확인.

In [ ]:
print("=== H4-b 주별 비교 ===")
compare_rates("신규: 블프 주 vs 마감 직전 주",   306, 754, 232, 361)
compare_rates("재방문: 블프 주 vs 마감 직전 주", 413, 524, 434, 512)

## 4-1. H4-b 보완: 12/15 이후 상승의 정체 (39번 쿼리, 신규)

A = 11/26~12/13 (18일), B = 12/15~12/18 (4일).
결제 정보 전환율은 크게 오르지만(42.0% → 70.5%), 결제를 시작하는 비율이 함께 떨어짐(47.4% → 34.2%).
하루 평균 결제 정보 도달은 47.7명 vs 46.0명으로 같음 → **개선이 아니라 분모(결제 시작) 감소**.

In [ ]:
print("=== H4-b 보완: 12/15 이후 변화 (신규) ===")
compare_rates("결제 시작 → 결제 정보 (상승)",  858, 2043, 184, 261)
compare_rates("구매 의도 → 결제 시작 (하락)", 2043, 4310, 261, 763)
print(f"하루 평균 결제 정보 도달: A {858/18:.1f}명 vs B {184/4:.1f}명")


## 5. 신규(A) vs 재방문(B), 결제 정보 단계 (31·39번 쿼리)

시즌 전체 44.8% vs 82.8%, 성수기(11/26~12/14) 42.3% vs 82.1% → 개선 대상 확정

In [ ]:
compare_rates("시즌 결제 정보 전환율: 신규 vs 재방문", 1232, 2749, 1992, 2405)
compare_rates("성수기 결제 정보 전환율: 신규 vs 재방문", 906, 2143, 1411, 1718)


## 6. A/B 표본 수와 가드레일 기준 (40·41번 쿼리 기준값)

- 대상: 성수기(11/26~12/14, 19일) 신규 중 장바구니 도달자
- 핵심 지표: **구매 ÷ 장바구니 도달** 14.4% (654 / 4,554), 하루 240명
- 결제 정보 ÷ 결제 시작(42.3%)은 개선안이 분모를 바꾸므로 진단 지표로만 사용
- α = 0.05, 검정력 = 0.8, 50:50 배정
- 결과: 19일이면 **14.4% → 17.4% (+3.0%p, 상대 +21%)** 이상의 효과를 검출 가능, 손익분기 객단가 하락 −17.4%

In [ ]:
import math
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline = 654 / 4554   # 성수기 신규: 장바구니 도달 → 구매 (14.4%, 40번)
daily = 240             # 성수기 신규 장바구니 도달 하루 평균 (4,554 / 19일)
days = 19               # 성수기 11/26 ~ 12/14

print("=== 효과 크기별 필요 표본 ===")
for lift in [0.01, 0.02, 0.03, 0.05]:
    target = baseline + lift
    es = proportion_effectsize(target, baseline)
    n_per_group = NormalIndPower().solve_power(
        effect_size=es, alpha=0.05, power=0.8, ratio=1, alternative='two-sided'
    )
    total = n_per_group * 2
    print(f"+{lift*100:.0f}%p (상대 +{lift/baseline*100:.0f}%) | "
          f"그룹당 {n_per_group:,.0f} | 합계 {total:,.0f} | 약 {total/daily:.1f}일")

# 성수기 19일로 잡을 수 있는 가장 작은 효과 (역계산)
n_per_group = daily * days / 2
es_min = NormalIndPower().solve_power(nobs1=n_per_group, alpha=0.05, power=0.8, ratio=1)
target_min = math.sin((es_min + 2 * math.asin(math.sqrt(baseline))) / 2) ** 2
rel_lift = target_min / baseline - 1
print(f"\n=== {days}일 동안 검출 가능한 최소 효과 ===")
print(f"그룹당 {n_per_group:,.0f} | {baseline:.1%} → {target_min:.1%} "
      f"(+{(target_min-baseline)*100:.1f}%p, 상대 +{rel_lift*100:.0f}%)")

# 가드레일 손익분기: 전환율이 최소 검출 효과만큼 오를 때 객단가가 얼마나 떨어져도 되는가
print(f"손익분기 객단가 하락: {1/(1+rel_lift)-1:.1%}")


In [ ]:
# 가드레일(주문당 객단가) 판정 기준 (41번: 주문 681건, 평균 65.76, 표준편차 72.94)
aov, sd, orders = 65.76, 72.94, 681

def drop_threshold(n_per_group, z):
    se_rel = sd * math.sqrt(2 / n_per_group) / aov
    return se_rel * z * 100

n_group = orders / 2
first_week = orders / days * 7 / 2
print(f"유의하게 나오는 최소 하락폭 : {drop_threshold(n_group, 1.96):.1f}%")
print(f"80% 확률로 잡는 하락폭     : {drop_threshold(n_group, 1.96 + 0.84):.1f}%")
print(f"첫 주(그룹당 약 {first_week:.0f}건) 유의 기준: {drop_threshold(first_week, 1.96):.1f}% → 조기 중단 기준 −30%")


## 7. 해석에 쓴 추가 계산

민감도 확인(퍼널 방식·비교 기준·기간 경계)은 SQL(30, 32, 33번)로 했습니다. 여기는 문서에 나오는 숫자 중 SQL 결과를 다시 계산해 만든 값을 재현합니다.

- 7-1. 장바구니를 쓰지 않는 4단계 하락 기여도와 "한 단계만 바꿨다면" 계산 (30번)
- 7-2. H3 재확인: 조회 → 결제 시작 기준 신규 vs 재방문 (31번)
- 7-3. KPI 트리를 로그로 나눈 이유: 순서에 따라 달라지는 단순 분해 (36번)
- 7-4. 국가 값 없음 15세션의 최악의 경우 (27·34번)
- 7-5. 정상 기간 장바구니 없는 결제 시작 비율 비교 (29번)

In [ ]:
# 7-1. 4단계 하락 기여도 (30번 최대 도달 기준 세션 수)
season = {'visit': 124133, 'view': 28454, 'checkout': 5154, 'payment': 3224, 'purchase': 2369}
off    = {'visit': 107189, 'view': 21514, 'checkout': 2038, 'payment': 1482, 'purchase': 1059}
steps = [('방문 → 조회', 'visit', 'view'), ('조회 → 결제 시작', 'view', 'checkout'),
         ('결제 시작 → 결제 정보', 'checkout', 'payment'), ('결제 정보 → 구매', 'payment', 'purchase')]

ratios = {name: (off[b]/off[a]) / (season[b]/season[a]) for name, a, b in steps}
total = sum(math.log(r) for r in ratios.values())
print("=== 하락 기여도 (장바구니 무관 4단계) ===")
for name, a, b in steps:
    print(f"{name:12s} 시즌 {season[b]/season[a]:6.1%} → 비시즌 {off[b]/off[a]:6.1%} | 기여 {math.log(ratios[name])/total:6.1%}")

cvr_season = season['purchase'] / season['visit']
print(f"\n=== 한 단계만 비시즌 값으로 바꿨다면 (시즌 세션 전환율 {cvr_season:.2%}) ===")
for name in ratios:
    print(f"{name:12s} → {cvr_season * ratios[name]:.2%}")
others = math.prod(r for n, r in ratios.items() if n != '조회 → 결제 시작')
print(f"나머지 세 단계 모두 → {cvr_season * others:.2%}  (실제 비시즌 {off['purchase']/off['visit']:.2%})")


In [ ]:
# 7-2. H3 재확인: 조회 → 결제 시작 (장바구니 무관)
# 조회 도달은 셀 5와 같고, 결제 시작 도달은 31번 단계 전환율로 계산한 세션 수
df2 = pd.DataFrame({
    'period':  ['season', 'season', 'off', 'off'],
    'segment': ['new', 'returning', 'new', 'returning'],
    'success': [2749, 2405, 942, 1096],      # 결제 시작 도달
    'n':       [18500, 9954, 15183, 6331],   # 조회 도달
})
df2['rate'] = df2['success'] / df2['n']
print(df2)
for seg in ['new', 'returning']:
    s = df2[df2.segment == seg].set_index('period')['rate']
    print(f"{seg}: {s['season']:.1%} → {s['off']:.1%} ({s['off']/s['season']-1:+.0%})")
m2 = smf.glm("rate ~ C(period, Treatment('season')) * C(segment, Treatment('returning'))",
             data=df2, family=sm.families.Binomial(), var_weights=df2['n']).fit()
inter = "C(period, Treatment('season'))[T.off]:C(segment, Treatment('returning'))[T.new]"
print(f"상호작용 오즈비 {np.exp(m2.params[inter]):.3f}, p = {m2.pvalues[inter]:.2e}")


In [ ]:
# 7-3. KPI 트리: 하나씩 바꾸는 분해는 순서에 따라 결과가 달라짐 (36번)
s_kpi = {'세션': 124133/26, '전환율': 2369/124133, '구매 세션당 매출': 185072/2369}
o_kpi = {'세션': 107189/27, '전환율': 1059/107189, '구매 세션당 매출': 53507/1059}
rev_s = math.prod(s_kpi.values()); rev_o = math.prod(o_kpi.values())
print(f"일 매출 {rev_s:,.0f} → {rev_o:,.0f} ({rev_o/rev_s-1:+.1%})")

def sequential(order):
    cur, out = dict(s_kpi), {}
    for k in order:
        before = math.prod(cur.values()); cur[k] = o_kpi[k]
        out[k] = (before - math.prod(cur.values())) / (rev_s - rev_o)
    return out

for order in [['세션', '전환율', '구매 세션당 매출'], ['구매 세션당 매출', '전환율', '세션']]:
    r = sequential(order)
    print("순서", " → ".join(order), "|", ", ".join(f"{k} {v:.0%}" for k, v in r.items()))

lt = math.log(rev_o / rev_s)
print("로그 분해 |", ", ".join(f"{k} {math.log(o_kpi[k]/s_kpi[k])/lt:.1%}" for k in s_kpi))


In [ ]:
# 7-4. 국가 값 없음 15세션의 최악의 경우 (34번 데스크톱·모바일 합계)
non_us_pay, non_us_chk = 663, 1464
us_pay, us_chk = 547, 1229
print(f"현재: 미국 외 {non_us_pay/non_us_chk:.1%} vs 미국 {us_pay/us_chk:.1%}")
print(f"15세션이 모두 결제 정보까지 갔다면 빼고: {(non_us_pay-15)/(non_us_chk-15):.1%}")
print(f"15세션이 모두 이탈했다면 빼고        : {non_us_pay/(non_us_chk-15):.1%}")


In [ ]:
# 7-5. 정상 기간 장바구니 없는 결제 시작 비율 (29번 쿼리 ①, 11/26~12/18 vs 1/5~1/31)
# 시즌: 결제 시작 4,545 중 장바구니 기록 있음 3,594 / 비시즌: 2,038 중 1,463
compare_rates("장바구니 없는 결제 시작: 시즌(정상 기간) vs 비시즌", 4545 - 3594, 4545, 2038 - 1463, 2038)


## 한계
- 세션 간 독립 가정: 재방문이 섞인 비교는 p값이 실제보다 약간 작을 수 있음 (신규는 사람당 한 세션)
- 사전에 정한 가설만 검정. 결론에 쓴 결과는 p < 0.0001이거나 p > 0.5로 판정이 흔들리지 않음
- 검정은 "차이가 우연이 아니다"까지. 이탈 이유(최종 금액 / 도착 예정일)는 로그로 구분할 수 없는 해석이며 A/B로 검증할 가설
- 객단가 가드레일은 검정력이 낮아(손익분기 수준 하락을 잡을 확률 약 50%) 장바구니 도달자당 매출과 함께 판단